In [1]:
%load_ext autoreload
%autoreload 2

# Import library

In [2]:
import pandas as pd
import numpy as np
from PIL import Image

# Set path

In [22]:
labels = '/project/lt200264-saiwat/WormProject/data/worm-24022026/labels/worm-24022026-labels_7class_100images.csv'
roi_metadata = '/project/lt200264-saiwat/WormProject/data/worm-24022026/processed/cropped/dataset_meta.pkl'
masks = '/project/lt200264-saiwat/WormProject/data/worm-24022026/processed/sam2_mask'

In [4]:
labels = pd.read_csv(labels)
labels

,Class,Filename
0,1,worm-000001_000.npy
1,1,worm-000001_001.npy
2,2,worm-000001_002.npy
3,1,worm-000001_003.npy
4,1,worm-000001_004.npy
...,...,...
31896,5,worm-000100_293.npy
31897,4,worm-000100_294.npy
31898,1,worm-000100_295.npy
31899,1,worm-000100_296.npy


In [6]:
metadata = pd.read_pickle(roi_metadata)
metadata

,stem,src_npy_path,mask_npy_path,bbox,roi_bbox_xyxy,idx,raw_img_path
0,worm-000001,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[2057.0, 2272.0, 92.0, 115.0]","(1851, 1943, 2356, 2448)",0,/project/lt200264-saiwat/WormProject/data/worm...
1,worm-000001,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[1292.0, 2249.0, 38.0, 131.0]","(1059, 1943, 1564, 2448)",1,/project/lt200264-saiwat/WormProject/data/worm...
2,worm-000001,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[2037.0, 1212.0, 163.0, 77.0]","(1866, 998, 2371, 1503)",2,/project/lt200264-saiwat/WormProject/data/worm...
3,worm-000001,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[1521.0, 1478.0, 132.0, 43.0]","(1335, 1247, 1840, 1752)",3,/project/lt200264-saiwat/WormProject/data/worm...
4,worm-000001,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[2250.0, 1484.0, 59.0, 105.0]","(2027, 1284, 2532, 1789)",4,/project/lt200264-saiwat/WormProject/data/worm...
...,...,...,...,...,...,...,...
70336,worm-000216,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[927.0, 1572.0, 44.0, 46.0]","(615, 1261, 1283, 1929)",326,/project/lt200264-saiwat/WormProject/data/worm...
70337,worm-000216,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[1925, 1460, 94, 65]","(1638, 1158, 2306, 1826)",327,/project/lt200264-saiwat/WormProject/data/worm...
70338,worm-000216,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[1441, 598, 81, 49]","(1147, 288, 1815, 956)",328,/project/lt200264-saiwat/WormProject/data/worm...
70339,worm-000216,/project/lt200264-saiwat/WormProject/data/worm...,/project/lt200264-saiwat/WormProject/data/worm...,"[1522, 590, 31, 59]","(1203, 285, 1871, 953)",329,/project/lt200264-saiwat/WormProject/data/worm...


In [14]:
stems = metadata['stem'].sort_values().unique()
stems

<StringArray>
['worm-000001', 'worm-000002', 'worm-000003', 'worm-000004', 'worm-000005',
 'worm-000006', 'worm-000007', 'worm-000008', 'worm-000009', 'worm-000010',
 ...
 'worm-000208', 'worm-000209', 'worm-000210', 'worm-000211', 'worm-000212',
 'worm-000213', 'worm-000214', 'worm-000215', 'worm-000216', 'worm-000217']
Length: 217, dtype: str

In [23]:
stem_paths = []
for stem in stems:
    path = f'{masks}/{stem}.pkl'
    stem_paths.append(path)

In [25]:
image_mask = pd.read_pickle(stem_paths[0])
image_mask

,segmentation,area,bbox,predicted_iou,point_coords,stability_score,crop_box,aspect_ratio,mask2box_ratio,R,...,B,H,S,V,L,A,BY,bUse,roi_bbox_xyxy,max_ratio
0,"[[False, False, False, False, False, False, Fa...",4285,"[2057.0, 2272.0, 92.0, 115.0]",0.964844,"[[2127.4453125, 2281.3359375]]",0.982168,"[1215.0, 807.0, 2049.0, 1641.0]",3.76,0.405009,92.0,...,94.0,69.0,32.0,101.0,106.0,122.0,131.0,1,None,1.250000
1,"[[False, False, False, False, False, False, Fa...",3661,"[1292.0, 2249.0, 38.0, 131.0]",0.960938,"[[1327.0546875, 2255.6953125]]",0.979719,"[1215.0, 807.0, 2049.0, 1641.0]",3.96,0.735436,93.0,...,95.0,77.0,17.0,99.0,105.0,126.0,129.0,1,None,3.447368
2,"[[False, False, False, False, False, False, Fa...",5108,"[2037.0, 1212.0, 163.0, 77.0]",0.957031,"[[2191.4765625, 1230.0703125]]",0.979868,"[1215.0, 807.0, 2049.0, 1641.0]",4.43,0.406980,79.0,...,100.0,93.0,55.0,102.0,103.0,121.0,125.0,1,None,2.116883
3,"[[False, False, False, False, False, False, Fa...",2859,"[1521.0, 1478.0, 132.0, 43.0]",0.957031,"[[1647.2109375, 1486.4765625]]",0.987843,"[1215.0, 807.0, 2049.0, 1641.0]",5.78,0.503700,114.0,...,127.0,83.0,39.0,134.0,138.0,120.0,128.0,1,None,3.069767
4,"[[False, False, False, False, False, False, Fa...",4110,"[2250.0, 1484.0, 59.0, 105.0]",0.957031,"[[2255.5078125, 1512.1171875]]",0.982634,"[1215.0, 807.0, 2049.0, 1641.0]",2.19,0.663438,77.0,...,88.0,79.0,42.0,94.0,97.0,121.0,129.0,1,None,1.779661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,"[[False, False, False, False, False, False, Fa...",1070,"[1741.0, 1670.0, 62.0, 30.0]",0.808594,"[[1772.25, 1673.4375]]",0.911607,"[0.0, 0.0, 3264.0, 2448.0]",2.73,0.575269,86.0,...,156.0,101.0,115.0,156.0,136.0,120.0,111.0,0,None,2.066667
276,"[[False, False, False, False, False, False, Fa...",1641,"[1614, 942, 31, 102]",0.902344,"[[1628, 992]]",0.894737,"[1533, 831, 248, 324]",5.19,0.518975,35.0,...,46.0,86.0,67.0,49.0,48.0,123.0,127.0,2,"[1533, 831, 1781, 1155]",3.290323
277,"[[False, False, False, False, False, False, Fa...",1075,"[1602, 1375, 37, 42]",0.859375,"[[1616, 1393]]",0.691310,"[1529, 1297, 141, 157]",1.41,0.691763,81.0,...,152.0,97.0,120.0,152.0,137.0,115.0,113.0,2,"[1529, 1297, 1670, 1454]",1.135135
278,"[[False, False, False, False, False, False, Fa...",682,"[1611, 1519, 35, 31]",0.902344,"[[1629, 1534]]",0.825160,"[1560, 1427, 205, 248]",1.19,0.628571,81.0,...,147.0,99.0,113.0,147.0,128.0,118.0,112.0,2,"[1560, 1427, 1765, 1675]",1.129032


In [46]:
def _xywh_to_xyxy(bbox): # private function
    x, y, w, h = bbox
    return [x, y, x + w, y + h]

In [50]:
def get_overlap_bbox(bboxes):
    bboxes = np.array([_xywh_to_xyxy(bbox) for bbox in bboxes], dtype=float) # แปลง list เป็น numpy array
    x1, y1, x2, y2 = bboxes[:, 0], bboxes[:, 1], bboxes[:, 2], bboxes[:, 3] # XYXY, np.array[row, column]

    # check overlap
    
    
    return

In [42]:
def a(roi_metadata, masks_dir):
    metadata = pd.read_pickle(roi_metadata)
    stems = metadata['stem'].sort_values().unique()

    for stem in stems:
        mask_path = f'{masks_dir}/{stem}.pkl'
        mask = pd.read_pickle(mask_path)

        overlap_bbox = get_overlap_bbox(mask['bbox'])

In [51]:
b = [[1292.0, 2249.0, 38.0, 131.0]]
c = get_overlap_bbox(b)
print(c)

[[1292.0, 2249.0, 1330.0, 2380.0]]
